# CAT 1 – Causes of Road Traffic Crashes in Uganda (2011–2020)
**Q2:** Import the cleaned data from my GitHub data warehouse and produce a heatmap of the
relationships among Population_Behavior, Road_Environment, Motor_Vehicle_Condition and
Total Road Traffic Accidents.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats

# Raw-file URLs from my GitHub data warehouse (bc7144-oss/uganda-road-crash-warehouse)
GITHUB_USER = "bc7144-oss"
REPO = "uganda-road-crash-warehouse"
BASE = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO}/main/data/"

def load(name):
    try:
        return pd.read_csv(BASE + name)
    except Exception as e:                       # offline fallback: local copy of the same file
        print(f"GitHub not reachable ({type(e).__name__}); reading local data/{name}")
        return pd.read_csv("data/" + name)

causes = load("uganda_crash_causes_clean_wide.csv")
factors = load("uganda_crashes_by_factor.csv")
factors

## How causes were grouped into the analysis variables
* **Population_Behavior** – careless driving, reckless driving, over-speeding, drink-driving,
  careless pedestrians, passengers falling from vehicles
* **Motor_Vehicle_Condition** – defective mechanical condition, over-loading, dangerous loading
* **Road_Environment** – obstacles on the carriageway, dazzling lights
* *Unknown* and *Other* causes are not attributed to a factor but are included in the Total.

In [ ]:
causes.groupby(["Factor", "Cause"]).Total.sum().sort_values(ascending=False).to_frame()

In [ ]:
VARS = ["Population_Behavior", "Road_Environment", "Motor_Vehicle_Condition",
        "Total_Road_Traffic_Accidents"]
LABELS = ["Population\nBehavior", "Road\nEnvironment", "Motor Vehicle\nCondition", "Total Road\nTraffic Accidents"]

corr = factors[VARS].corr(method="pearson")
print(corr.round(2))

# p-values for each pair (n = 10 years)
pvals = pd.DataFrame(np.ones((4, 4)), index=VARS, columns=VARS)
for a in VARS:
    for b in VARS:
        if a != b:
            pvals.loc[a, b] = stats.pearsonr(factors[a], factors[b])[1]
print(pvals.round(4))

## Heatmap (Pearson correlation, 2011–2020)

In [ ]:
# diverging blue -> neutral gray -> red, so 0 reads as "no relationship"
cmap = LinearSegmentedColormap.from_list("div", ["#2a78d6", "#f0efec", "#c9382f"])

annot = corr.round(2).astype(str)
for a in VARS:
    for b in VARS:
        if a != b:
            p = pvals.loc[a, b]
            annot.loc[a, b] += "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else ""

fig, ax = plt.subplots(figsize=(7.5, 6))
sns.heatmap(corr, annot=annot, fmt="", cmap=cmap, vmin=-1, vmax=1, center=0, square=True,
            linewidths=2, linecolor="white", cbar_kws={"label": "Pearson r", "shrink": .8},
            annot_kws={"size": 12, "color": "#0b0b0b"},
            xticklabels=LABELS, yticklabels=LABELS, ax=ax)
ax.set_title("Correlation of crash factors with total road traffic crashes\nUganda, 2011–2020 (n = 10 years)",
             fontsize=12, loc="left", pad=12)
ax.tick_params(axis="x", rotation=0, labelsize=9)
ax.tick_params(axis="y", rotation=0, labelsize=9)
fig.text(0.01, 0.01, "* p<0.05   ** p<0.01   *** p<0.001", fontsize=8, color="#52514e")
fig.tight_layout()
fig.savefig("heatmap_correlation.png", dpi=200)
plt.show()

## Supporting view: trends and shares

In [ ]:
share = factors.set_index("Year")[VARS[:3]].div(factors.set_index("Year").Total_Road_Traffic_Accidents, axis=0) * 100
print(share.round(1))

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(factors.Year, factors.Total_Road_Traffic_Accidents, color="#0b0b0b", lw=2, marker="o", label="Total crashes")
ax.plot(factors.Year, factors.Population_Behavior, color="#c9382f", lw=2, marker="o", label="Population behavior")
ax.plot(factors.Year, factors.Motor_Vehicle_Condition, color="#2a78d6", lw=2, marker="o", label="Motor vehicle condition")
ax.plot(factors.Year, factors.Road_Environment, color="#1f9e89", lw=2, marker="o", label="Road environment")
ax.set_ylabel("Crashes per year"); ax.set_xticks(factors.Year)
ax.grid(axis="y", color="#e5e4e0"); ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=9)
ax.set_title("Road traffic crashes by contributing factor, Uganda 2011–2020", loc="left", fontsize=12)
fig.tight_layout(); fig.savefig("trend_by_factor.png", dpi=200); plt.show()

## Robustness: Spearman rank correlation and dropping 2011
2011 is an outlier for road-environment and vehicle causes, and n is only 10, so check
whether the conclusions depend on it.

In [ ]:
print("Spearman:\n", factors[VARS].corr("spearman").round(2))
print("\nPearson, 2012–2020 only:\n", factors[factors.Year > 2011][VARS].corr().round(2))